### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [2]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[55])

I sent a response to the White House at

	0005895485@MCIMAIL.COM (White House)

and received a nice, automatic reply from MICMAIL noting, in passing, that
if I had included a SNail address, I would get a reply in due course.

For those who care, my reply was:

	1.	yes, let's protect the voice network

	2.	privately-developed crypto has always been available and
		always will be -- so let's think about how to do law
		enforcement given that fact not about how to hope to
		legislate against it

	3.	my needs for crypto as a system designer are not met by the
		Clipper Chip.  I want freely to export uses of algorithms
		(like DES & RSA) which are already freely available in the
		destination country



Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Resolución del desafío
### Paso 1.1 Elegir 5 documentos al azar

Primero fijo una semilla para que el experimento sea reproducible.

In [12]:
import numpy as np

np.random.seed(123)
n_docs = X_train.shape[0]  # cantidad total de documentos en train
indices_random = np.random.choice(n_docs, size=5, replace=False)
indices_random

array([  872, 10976,  2573, 11214, 10939], dtype=int32)

### Paso 1.2 — Calcular similaridad de un documento contra todo el corpus

In [13]:
doc_idx = indices_random[0]  # arrancamos con el primero de los 5
sims = cosine_similarity(X_train[doc_idx], X_train).flatten()
sims.shape

(11314,)

X_train[doc_idx] es una matriz de 1 fila (el documento elegido). cosine_similarity la compara contra cada fila de X_train, así que sims termina siendo un vector con un valor de similaridad por cada documento del corpus.


### Paso 1.3 Ordeno y me quedo con los 5 más similares (excluyendo el propio documento)

In [14]:
# argsort ordena de menor a mayor, así que invierto con [::-1]
indices_ordenados = np.argsort(sims)[::-1]

# el primero de la lista va a ser el propio documento (similaridad = 1.0), lo saco
top5_similares = indices_ordenados[1:6]
top5_similares, sims[top5_similares]

(array([ 9623,  2009, 10123,  7953,  2477]),
 array([0.17691697, 0.17326366, 0.16949712, 0.16690533, 0.16204287]))

### Paso 1.4 Mirar el contenido y la clase de esos documentos

In [15]:
print("="*80)
print(f"DOCUMENTO ORIGINAL (índice {doc_idx})")
print(f"Clase: {newsgroups_train.target_names[newsgroups_train.target[doc_idx]]}")
print("-"*80)
print(newsgroups_train.data[doc_idx][:500])  # primeros 500 caracteres
print("="*80)

for rank, idx in enumerate(top5_similares, start=1):
    print(f"\n--- Vecino #{rank} (índice {idx}) | similaridad = {sims[idx]:.4f} ---")
    print(f"Clase: {newsgroups_train.target_names[newsgroups_train.target[idx]]}")
    print(newsgroups_train.data[idx][:300])

DOCUMENTO ORIGINAL (índice 872)
Clase: alt.atheism
--------------------------------------------------------------------------------


But, you wouldn't know what red *was*, and you certainly couldn't judge
it subjectively.  And, objectivity is not applicable, since you are wanting
to discuss the merits of red.

--- Vecino #1 (índice 9623) | similaridad = 0.1769 ---
Clase: talk.politics.mideast
Accounts of Anti-Armenian Human Right Violations in Azerbaijan #012
                 Prelude to Current Events in Nagorno-Karabakh

        +---------------------------------------------------------+
        |                                                         |
        |  I saw a naked girl wi

--- Vecino #2 (índice 2009) | similaridad = 0.1733 ---
Clase: sci.crypt
:Judge: "I grant you immunity from whatever may be learned from the key
:	itself"
:You:    "The keyphrase is: "I confess to deliberately evading copyright; 
:	the file encoded with this keyphrase contains illegal scans of 
:     

Interpretación caso 1:

El documento original es muy corto y usa vocabulario abstracto/genérico ("red", "subjetivo", "objetivo", "juzgar"), no términos específicos de ateísmo. Esto resulta en que ninguno de los 5 vecinos pertenece a la clase alt.atheism, y las similaridades son bajas (0.16–0.18, muy lejos de 1).

Por otro lado, mirando el contenido: el vecino #3 (talk.religion.misc) también discute qué es "objetivo" vs "subjetivo" en un contexto filosófico/religioso. Ahí sí hay una relación temática real, más allá de que la etiqueta no coincida exactamente (son categorías vecinas: religión vs ateísmo, ambas tocan filosofía).

Los demás vecinos (baseball, criptografía, política) comparten con el original solo palabras sueltas como "red" (que en inglés aparece tanto en "Red Sox" y "red blood clot" como en el sentido de color). Aquí se observa un caso de colisión léxica sin relación semántica real: TF-IDF pondera palabras, no significado, así que un término ambiguo o muy común puede acercar documentos que no tienen nada que ver temáticamente.


### Paso 1.5 Armo el loop para los 4 documentos restantes

In [ ]:
def mostrar_similares(doc_idx, X, corpus, top_n=5):
    sims = cosine_similarity(X[doc_idx], X).flatten()
    indices_ordenados = np.argsort(sims)[::-1]
    top_similares = indices_ordenados[1:top_n+1]

    print("="*80)
    print(f"DOCUMENTO ORIGINAL (índice {doc_idx})")
    print(f"Clase: {corpus.target_names[corpus.target[doc_idx]]}")
    print("-"*80)
    print(corpus.data[doc_idx][:500])
    print("="*80)

    for rank, idx in enumerate(top_similares, start=1):
        print(f"\n--- Vecino #{rank} (índice {idx}) | similaridad = {sims[idx]:.4f} ---")
        print(f"Clase: {corpus.target_names[corpus.target[idx]]}")
        print(corpus.data[idx][:300])
    print("\n")

# Caso 2 

mostrar_similares(indices_random[1], X_train, newsgroups_train)

DOCUMENTO ORIGINAL (índice 10976)
Clase: alt.atheism
--------------------------------------------------------------------------------

Sorry.   Wrong.    This is how banks got started in the first place.
Sooner or later your father and his pals will lend money to someone
who eventually goes broke, and then they will realise that they
havn't been managing risk very well.   Then they will ask themselves
what it is that they need to quantify risk, and to persuade borrowers
not to take on greater loans than they can carry.    And since they
don't all want the worry of doing the calculations and handling the
money, some of them wil

--- Vecino #1 (índice 6437) | similaridad = 0.3175 ---
Clase: talk.politics.mideast
Accounts of Anti-Armenian Human Rights Violations in Azerbaijan #007
                 Prelude to Current Events in Nagorno-Karabakh


 +--------------------------------------------------------------------------+
 |                                                                  

Interpretación caso 2:

El documento original está catalogado como alt.atheism, pero el contenido no tiene nada que ver con religión: habla de bancos, préstamos y gestión de riesgo.

Los vecinos #1 a #4 son casi el mismo documento repetido: son las partes A, B, 007, 008, 012 de una misma serie ("Accounts of Anti-Armenian Human Rights Violations in Azerbaijan"). Tiene sentido que sean altamente similares entre sí, pero no necesariamente con el documento original sobre bancos. Si bien las similaridades acá son más altas que en el caso anterior, probablemente no sea por relación temática con el original, sino porque esas 4 partes comparten formato (los bordes de asteriscos +---+, el título repetido "Prelude to Current Events...") que el vectorizador toma como texto normal y les da peso.

El vecino #5 (talk.religion.misc, sobre Waco/Koresh) es el único con algo más cercano temáticamente al original en tanto ambos tocan crítica social/institucional, aunque tampoco es sobre bancos específicamente.

En resumen, este caso ilustra dos problemas distintos del enfoque TF-IDF + similaridad coseno: 
* El ruido de etiqueta (el documento está mal clasificado respecto a su contenido real). 
* Documentos casi duplicados entre sí (partes de una misma serie) pueden "dominar" el ranking de similaridad sin que eso diga nada sobre el documento consultado.

In [ ]:
# Caso 3
mostrar_similares(indices_random[2], X_train, newsgroups_train)

DOCUMENTO ORIGINAL (índice 2573)
Clase: talk.politics.mideast
--------------------------------------------------------------------------------

While that is currently true from their perspective, it is also
worthwhile to note that in such cases the populace often does suffer
from attempts to control the guerillas.  Furthermore, there were
cases in the past of Palestinian gun emplacements being situated
within villages.  The argument that can be made for small arms
fire can not be made for field pieces.


As I recall, Amal was primarily nationalistically "Lebanon for
the Lebanese" motivated.  I think that the difference between them
wa

--- Vecino #1 (índice 9222) | similaridad = 0.4676 ---
Clase: talk.politics.mideast


The village I described was actually the closest I could come to
describing mine.  I agree there may be other villages where the civilian
population has deserted because it is too close to Israeli lines and
thus gets bombed more often.  In such villages often the only 

Inteprestación caso 3:

Las 5 clases coinciden exactamente con la del original: talk.politics.mideast.

Las similaridades son bastante más altas que en los dos casos anteriores (0.39–0.47), y hay coherencia temática real: todos los vecinos discuten el conflicto en el sur del Líbano, guerrilleros, población civil, tropas israelíes/sirias.

A diferencia de los casos anteriores, acá no hay señales de "colisión léxica" ni de series repetidas: son mensajes distintos de un mismo hilo de debate, con vocabulario compartido genuinamente relacionado al tema (Lebanon, villages, guerillas, Israeli, civilians).

El documento original es más largo y usa vocabulario específico del dominio (nombres de facciones, términos militares/políticos), lo cual da vectores TF-IDF más "informativos" y discriminantes que en los casos de documentos cortos y genéricos que se vieron anteriormente.

En conclusión, la similaridad coseno sobre TF-IDF funciona bien cuando el documento tiene suficiente longitud y vocabulario específico del tema, porque ahí los términos de mayor peso (IDF alto) realmente anclan el vector a ese dominio semántico.

In [20]:
# Caso 4

mostrar_similares(indices_random[3], X_train, newsgroups_train)

DOCUMENTO ORIGINAL (índice 11214)
Clase: rec.sport.hockey
--------------------------------------------------------------------------------


Nonsense.  I quite clearly state that it was Greg that made the claim
that Gainey never made an error.  And he made the claim. Read below.

From rec.sport.hockey Thu Apr 15 21:22:49 1993
From: gballent@hudson.UVic.CA (Greg  Ballentine)
Message-ID: <1993Apr15.160450.27799@sol.UVic.CA>

[nonsense deleted]

Gainey is the best defensive forward ever.  I stand by that assessment.
He was a very good player who belongs in the hall of fame.  Did you
ever watch him play? He never made a technical error

--- Vecino #1 (índice 8363) | similaridad = 0.3914 ---
Clase: rec.sport.hockey


Perhaps it was Trottier.  It happened behind the Habs goal if I recall.
Gainey simply didn't have his head up as he was picking up the puck.


If Gilmour was taken completely by surprise, as Gainey was, then yeah,
I would have to say that Doug wasn't playing "technically" smart

Interpretación caso 4:

Los vecinos #1 a #4 son todos rec.sport.hockey y con clara coherencia temática: todos discuten el mismo debate específico dentro del hilo (si Gainey cometió o no un error técnico, comparaciones con Trottier/Gilmour/Lemieux, el trofeo Selke). Existen similaridades altas y consistentes (0.34–0.39), similar al caso anterior de Medio Oriente: documento largo, vocabulario específico del dominio (nombres de jugadores, terminología de hockey), buen anclaje semántico.

Sin embargo el vecino #5 rompe el patrón: es soc.religion.christian, sobre genocidio y la voluntad de Dios. Temáticamente no tiene relación con hockey. Lo que probablemente los acerca es la estructura discursiva compartida, no el contenido: ambos documentos son citas de discusión tipo "fulano escribió... yo respondo...", con signos de cita (>), nombres propios entre paréntesis, y vocabulario de debate/argumentación ("claim", "wrote", "reply", "worthy/unworthy"). Es otro caso de colisión léxica, pero esta vez por estilo de escritura (formato de foro/debate) en lugar de por palabras ambiguas como en el primer caso.

In [21]:
# Caso 5 

mostrar_similares(indices_random[4], X_train, newsgroups_train)

DOCUMENTO ORIGINAL (índice 10939)
Clase: rec.autos
--------------------------------------------------------------------------------


A list of options that would be useful. They can be existing
options on a car, or things you'd like to have...

1) Tripmeter, great little gadget. Lets you keep rough track of>
   mileage, makes a good second guesser for your gas gauge...

2) Full size spare

3) Built in mountings and power systems for radar detectors.

4) a fitting that allows you to generate household current with
the engine running, and plug ins in the trunk, engine compartment
and cabin.

Feel free to add on...

5) Power w

--- Vecino #1 (índice 4659) | similaridad = 0.9748 ---
Clase: rec.autos
A list of options that would be useful. They can be existing
options on a car, or things you'd like to have...

1) Tripmeter, great little gadget. Lets you keep rough track of
   mileage, makes a good second guesser for your gas gauge...

2) Full size spare

3) Built in mountings and power sys

Interpretación caso 5:

El vecino #1 tiene una similaridad de 0.9748, prácticamente es un documento duplicado (mismo texto, misma lista de opciones de auto). Esto podría sea un repost o un thread donde alguien citó/reenvió el mensaje original casi textual.

Las 5 clases coinciden exactamente con rec.autos. El vecino #2 comparte varios ítems literales de la lista (el mismo punto sobre "generar corriente doméstica con el motor"), el #3 es otro mensaje del mismo hilo ("pointless options" — claramente la continuación de la conversación), y los últimos dos ya son sobre autos en general pero de temas distintos (preguntas frecuentes, un Porsche 914 vs Fiero).

Este documento es un caso donde el vocabulario es muy específico y técnico del dominio (partes de auto, gadgets), lo que genera vectores TF-IDF muy discriminantes, sin ambigüedad léxica ni problemas de formato como en los casos anteriores.

### Paso 2.1 Vectorizo la base test con el vectorizador ya ajustado en train

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

### Paso 2.2 Calculo la matriz de similaridad test-vs-train

In [23]:
similarity_matrix = cosine_similarity(X_test, X_train)
similarity_matrix.shape

(7532, 11314)

### Paso 2.3 Busco, para cada fila (cada documento de test), el índice del documento de train con mayor similaridad, y le asigno esa clase.

In [24]:
indices_mas_similares = np.argmax(similarity_matrix, axis=1)
y_pred_zero_shot = newsgroups_train.target[indices_mas_similares]

Paso 2.4 Evaluo el clasificador por prototipos con F1-macro

Evaluo las predicciones y_pred_zero_shot contra las etiquetas reales, igual que se hizo con Naive Bayes en el ejemplo:

In [25]:
from sklearn.metrics import f1_score

f1_zero_shot = f1_score(y_test, y_pred_zero_shot, average='macro')
f1_zero_shot

0.5049911553681621

El enfoque de prototipos (1-NN con similaridad coseno) performa peor que Naive Bayes, aunque no por mucho. Esto tiene sentido porque Naive Bayes aprende una distribución de probabilidad por clase sobre todo el vocabulario (usa la información de las ~11.000 documentos de train de forma agregada por clase). Mientras que el clasificador por prototipos, decide la clase mirando un único vecino (el más cercano). Esto lo hace mucho más sensible al ruido: si ese vecino más cercano resulta ser un caso "raro" como los que se presentaron en el punto 1, la predicción se equivoca por completo, sin ningún mecanismo de "votación" o promediado que lo compense.


Una una posible mejora sería usar k>1 vecinos con votación, en vez de 1-NN puro, ya que eso suavizaría el efecto de los casos ruidosos.

### Paso 3.2 Función de entrenamiento y evaluación rápida

Encapsulo todo el flujo (vectorizar train, vectorizar test, entrenar el modelo, predecir, evaluar) en una sola función, para poder llamarla con distintas combinaciones de parámetros sin repetir código:

In [27]:
def evaluar_modelo(vectorizer, modelo, train_data, train_target, test_data, test_target):
    X_train = vectorizer.fit_transform(train_data)
    X_test = vectorizer.transform(test_data)

    modelo.fit(X_train, train_target)
    y_pred = modelo.predict(X_test)

    f1 = f1_score(test_target, y_pred, average='macro')
    return f1

### Paso 3.3 Primera prueba: baseline para confirmar que la función funciona igual que el ejemplo

In [28]:
f1_baseline = evaluar_modelo(
    vectorizer=TfidfVectorizer(),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)
f1_baseline

0.5854345727938506

Coincide exactamente con el baseline del ejemplo. La función está bien armada y lista para iterar.

### Paso 3.4 Pruebo variaciones

Antes de combinar todo, tiene sentido ir cambiando un parámetro por vez para entender el efecto individual de cada uno:

In [29]:
resultados = {}

resultados['baseline'] = f1_baseline

resultados['stop_words'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english'),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['sublinear_tf'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(sublinear_tf=True),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['stop_words_sublinear'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', sublinear_tf=True),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados

{'baseline': 0.5854345727938506,
 'stop_words': 0.6467991505900852,
 'sublinear_tf': 0.5859647447228136,
 'stop_words_sublinear': 0.6391030395780197}

Observaciones:

* stop_words='english' sola da el salto más grande. Al eliminar palabras funcionales (the, is, and, of...), que aparecen en todas las clases por igual, Naive Bayes deja de "gastar" probabilidad en términos que no discriminan nada entre categorías, y las palabras temáticamente relevantes pesan relativamente más en el cálculo.

* sublinear_tf=True sola casi no cambia nada (0.5860, prácticamente igual al baseline). Esto sugiere que en este dataset la repetición de palabras dentro de un mismo documento no es un factor que esté confundiendo mucho al modelo.

* La combinación de ambos (stop_words_sublinear = 0.6391) da peor resultado que stop_words sola (0.6468). Una posible explicación es que al quitar las stop words, el vocabulario que queda ya es mayormente discriminante, y aplicar además sublinear_tf (que atenúa términos repetidos) puede estar "aplanando" de más justo las palabras clave que sí conviene que pesen fuerte cuando se repiten dentro de un documento relevante.

Mantengo stop_words='english'.

### Paso 3.5 Sigo iterando: max_df/min_df y alpha

In [30]:
resultados['max_df'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', max_df=0.5),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['min_df'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', min_df=2),
    modelo=MultinomialNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['alpha_bajo'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english'),
    modelo=MultinomialNB(alpha=0.1),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['complement_nb'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english'),
    modelo=ComplementNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados

{'baseline': 0.5854345727938506,
 'stop_words': 0.6467991505900852,
 'sublinear_tf': 0.5859647447228136,
 'stop_words_sublinear': 0.6391030395780197,
 'max_df': 0.6467991505900852,
 'min_df': 0.6511573382063233,
 'alpha_bajo': 0.6725863920128103,
 'complement_nb': 0.6936107849650025}

Observaciones: 

* max_df=0.5 dio exactamente el mismo valor que stop_words sola (0.6468). Esto tiene sentido: en este dataset, prácticamente ninguna palabra de contenido aparece en más del 50% del total de los documentos de train, así que ese filtro no está sacando nada adicional a lo que ya sacaban las stop words.
* min_df=2 mejora un poco más (0.6512), la mejor mejora marginal hasta ahora sobre solo stop_words. Esto sugiere que sí había ruido real en palabras que aparecían una sola vez en todo el corpus (nombres propios, errores de tipeo, términos únicos) y sacarlas ayuda al modelo a generalizar mejor.
* alpha=0.1 en MultinomialNB mejora bastante más (0.6726) respecto al alpha por default. Esto indica que el suavizado de Laplace estándar (alpha=1.0) estaba "aplanando" demasiado las probabilidades ya que con un vocabulario de miles de términos, sumarle 1 a cada conteo tiene un efecto proporcionalmente grande cuando las frecuencias reales son bajas, así que bajar alpha deja que el modelo confíe más en lo que realmente observó.
* ComplementNB es el mejor resultado hasta ahora: 0.6936.

### Paso 3.6 Combino las mejores opciones

In [31]:
resultados['combo_1'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', min_df=2),
    modelo=ComplementNB(),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['combo_2'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', min_df=2),
    modelo=ComplementNB(alpha=0.1),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados['combo_3'] = evaluar_modelo(
    vectorizer=TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
    modelo=ComplementNB(alpha=0.1),
    train_data=newsgroups_train.data,
    train_target=newsgroups_train.target,
    test_data=newsgroups_test.data,
    test_target=newsgroups_test.target
)

resultados

{'baseline': 0.5854345727938506,
 'stop_words': 0.6467991505900852,
 'sublinear_tf': 0.5859647447228136,
 'stop_words_sublinear': 0.6391030395780197,
 'max_df': 0.6467991505900852,
 'min_df': 0.6511573382063233,
 'alpha_bajo': 0.6725863920128103,
 'complement_nb': 0.6936107849650025,
 'combo_1': 0.6942920490839366,
 'combo_2': 0.6886612228461628,
 'combo_3': 0.6878361252296743}

Observaciones:

* combo_1 (stop_words + min_df=2, ComplementNB con alpha default) da el mejor resultado hasta ahora: 0.6943, apenas por encima del ComplementNB "solo" (0.6936). Esto confirma que min_df=2 sigue aportando un poquito incluso cambiando de modelo, aunque la ganancia es marginal.
* combo_2 (agregando alpha=0.1) empeora a 0.6887. El alpha óptimo no es una propiedad del dataset en abstracto, sino que depende de cómo cada modelo usa las frecuencias. ComplementNB calcula sus estadísticos de forma distinta a MultinomialNB (usa los conteos del complemento de cada clase, no de la clase misma), así que el suavizado que necesita es diferente. Bajar alpha de más en ComplementNB probablemente hace que el modelo sobreajuste a particularidades del train.
* combo_3 (sumando sublinear_tf) empeora un poco más (0.6878). En este dataset, atenuar la frecuencia repetida dentro de un documento no ayuda, y en este modelo en particular parece incluso perjudicar levemente.

Se puede conlcuir que la optimización de hiperparámetros no es aditiva ni transferible entre modelos por lo que hay que explorar el espacio de cada modelo por separado, no asumir que la mejor configuración de uno sirve para el otro.

### Paso 3.7 — Afinar alpha de ComplementNB alrededor del default

In [32]:
for alpha_valor in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    resultados[f'complement_alpha_{alpha_valor}'] = evaluar_modelo(
        vectorizer=TfidfVectorizer(stop_words='english', min_df=2),
        modelo=ComplementNB(alpha=alpha_valor),
        train_data=newsgroups_train.data,
        train_target=newsgroups_train.target,
        test_data=newsgroups_test.data,
        test_target=newsgroups_test.target
    )

{k: v for k, v in resultados.items() if 'complement_alpha' in k}

{'complement_alpha_0.3': 0.6958233272814409,
 'complement_alpha_0.5': 0.697363066003392,
 'complement_alpha_0.7': 0.6967995995145294,
 'complement_alpha_1.0': 0.6942920490839366,
 'complement_alpha_1.5': 0.6914564143715054,
 'complement_alpha_2.0': 0.6893600555464913}

Conclusión: ComplementNB superó consistentemente a MultinomialNB (esperable por el desbalance de clases), el preprocesamiento del vectorizador (stop_words + min_df) aportó una mejora sólida e independiente del modelo, y el ajuste fino de alpha dio la última mejora, aunque marginal comparada con los cambios estructurales anteriores.


## Punto 4: Similaridad entre palabras
### Paso 4.1 Transpongo la matriz

In [33]:
X_train_terminos = X_train.T
X_train_terminos.shape

(101631, 11314)

Armo el vocabulario y una función para ubicar el índice de una palabra:

In [35]:
vocabulario = tfidfvect.get_feature_names_out()

def obtener_indice_palabra(palabra, vocab):
    indices = np.where(vocab == palabra)[0]
    if len(indices) == 0:
        raise ValueError(f"La palabra '{palabra}' no está en el vocabulario")
    return indices[0]

# probamos con una palabra
obtener_indice_palabra("hockey", vocabulario)

np.int64(47021)

Busco palabras

In [43]:
frecuencia_documentos = (X_train > 0).sum(axis=0)  # en cuántos documentos aparece cada palabra
frecuencia_documentos = np.asarray(frecuencia_documentos).flatten()

top_indices = np.argsort(frecuencia_documentos)[::-1][350:450]
for idx in top_indices:
    print(vocabulario[idx], frecuencia_documentos[idx])

mind 386
instead 384
key 384
25 383
simply 383
buy 383
feel 382
matter 382
saying 380
money 379
show 379
unless 376
later 376
making 375
price 373
book 373
important 373
news 373
similar 372
answer 370
certainly 369
tried 369
guess 368
understand 368
although 367
word 367
including 367
issue 367
working 366
nice 366
100 365
info 364
source 364
anybody 364
area 361
1993 358
address 358
told 357
wouldn 356
everyone 356
everything 356
side 355
known 355
11 355
team 355
often 354
net 353
home 352
advance 350
type 349
live 348
sort 345
phone 345
consider 344
except 344
standard 344
check 343
fine 341
cause 339
low 338
message 337
speed 336
pc 335
within 334
open 333
simple 332
goes 331
code 331
systems 331
couple 330
especially 329
posting 329
cost 328
usually 328
sorry 326
play 326
provide 326
university 325
exactly 324
correct 322
rest 322
memory 320
interesting 320
national 319
likely 319
john 317
claim 317
posted 316
files 316
deal 316
major 313
difference 313
machine 313
24 313
write 3

### Paso 4.3 Ubic0 los índices de esas 5 palabras en el vocabulario

In [44]:
palabras_elegidas = ["computer", "person", "university", "message", "team"]

def obtener_indice_palabra(palabra, vocab):
    indices = np.where(vocab == palabra)[0]
    if len(indices) == 0:
        raise ValueError(f"La palabra '{palabra}' no está en el vocabulario")
    return indices[0]

indices_palabras = [obtener_indice_palabra(p, vocabulario) for p in palabras_elegidas]
list(zip(palabras_elegidas, indices_palabras))

[('computer', np.int64(28940)),
 ('person', np.int64(70834)),
 ('university', np.int64(92213)),
 ('message', np.int64(61072)),
 ('team', np.int64(87968))]

### Paso 4.4 Calculo similaridad entre una palabra y todas las demás

In [45]:
def mostrar_palabras_similares(palabra, X_terminos, vocab, top_n=5):
    idx = obtener_indice_palabra(palabra, vocab)
    sims = cosine_similarity(X_terminos[idx], X_terminos).flatten()
    indices_ordenados = np.argsort(sims)[::-1]
    top_similares = indices_ordenados[1:top_n+1]  # salteamos la propia palabra

    print(f"Palabra: '{palabra}'")
    for rank, i in enumerate(top_similares, start=1):
        print(f"  {rank}. {vocab[i]}  (similaridad = {sims[i]:.4f})")
    print()

for palabra in palabras_elegidas:
    mostrar_palabras_similares(palabra, X_train_terminos, vocabulario)

Palabra: 'computer'
  1. decwriter  (similaridad = 0.1563)
  2. deluged  (similaridad = 0.1522)
  3. harkens  (similaridad = 0.1522)
  4. shopper  (similaridad = 0.1443)
  5. the  (similaridad = 0.1361)

Palabra: 'person'
  1. that  (similaridad = 0.1981)
  2. specificed  (similaridad = 0.1953)
  3. scotyy  (similaridad = 0.1953)
  4. of  (similaridad = 0.1933)
  5. rephrase  (similaridad = 0.1922)

Palabra: 'university'
  1. avigdor  (similaridad = 0.3080)
  2. nubar  (similaridad = 0.3080)
  3. masson  (similaridad = 0.3080)
  4. farah  (similaridad = 0.3080)
  5. chester  (similaridad = 0.3080)

Palabra: 'message'
  1. delightful  (similaridad = 0.1610)
  2. neglected  (similaridad = 0.1402)
  3. diffy  (similaridad = 0.1401)
  4. 7033d  (similaridad = 0.1381)
  5. error  (similaridad = 0.1372)

Palabra: 'team'
  1. teams  (similaridad = 0.2634)
  2. player  (similaridad = 0.2254)
  3. nhl  (similaridad = 0.2247)
  4. win  (similaridad = 0.1957)
  5. players  (similaridad = 0.1910)


Observaciones:

team es el único caso limpio: sus vecinos son teams, player, nhl, win, players. Posee coherencia semántica total, todo relacionado a deportes.
computer, person, message dan vecinos sin sentido: palabras  extrañas (decwriter, harkens, scotyy, diffy) mezcladas con stop words (the, that, of).
university es el caso más llamativo: 5 palabras distintas con exactamente la misma similaridad (0.3080). 

Esto sucede porque el vector de cada palabra vive en un espacio de ~11.000 dimensiones (una por documento de train), y la gran mayoría de esas dimensiones son cero para casi todas las palabras (sparsity extrema). Cuando una palabra tiene muy baja frecuencia de documento (aparece en 1 o 2 documentos nada más), su vector es casi todo ceros excepto en esas pocas posiciones. Si otra palabra rara aparece en esos mismos documentos, la similaridad coseno entre ambas puede dar artificialmente alta, aunque no tengan ninguna relación semántica real. Esto explica por qué university "empata" con 5 palabras random: probablemente las 6 palabras (incluida university) aparecen juntas en el mismo grupo pequeño de documentos (quizás firmas de email tipo "University of X").

Como no se filtró stop words en este vectorizador, esas palabras aparecen en casi todos los documentos, así que su vector tiene valores bajos pero repartidos en casi todas las dimensiones  por lo que terminan teniendo algo de similaridad con casi cualquier palabra, sin que eso signifique nada.

### Paso 4.5 — Corrijo el problema: filtrar vocabulario poco frecuente

Se propone construir la matriz término-documento con un vectorizador que saque stop words y exija una frecuencia mínima de documento, para que solo comparemos palabras con suficiente "señal".

In [46]:
tfidf_palabras = TfidfVectorizer(stop_words='english', min_df=10)
X_train_filtrado = tfidf_palabras.fit_transform(newsgroups_train.data)
vocabulario_filtrado = tfidf_palabras.get_feature_names_out()

X_train_terminos_filtrado = X_train_filtrado.T
X_train_terminos_filtrado.shape

(10441, 11314)

In [47]:
for palabra in palabras_elegidas:
    mostrar_palabras_similares(palabra, X_train_terminos_filtrado, vocabulario_filtrado)

Palabra: 'computer'
  1. wu  (similaridad = 0.1218)
  2. graphics  (similaridad = 0.1124)
  3. drive  (similaridad = 0.1067)
  4. incremental  (similaridad = 0.1065)
  5. reboot  (similaridad = 0.1049)

Palabra: 'person'
  1. people  (similaridad = 0.1537)
  2. presumed  (similaridad = 0.1388)
  3. incapable  (similaridad = 0.1347)
  4. smoked  (similaridad = 0.1042)
  5. implicitly  (similaridad = 0.1020)

Palabra: 'university'
  1. professor  (similaridad = 0.2480)
  2. department  (similaridad = 0.1977)
  3. edu  (similaridad = 0.1847)
  4. boulder  (similaridad = 0.1744)
  5. urbana  (similaridad = 0.1612)

Palabra: 'message'
  1. error  (similaridad = 0.1304)
  2. encrypted  (similaridad = 0.1241)
  3. neglected  (similaridad = 0.1150)
  4. kindly  (similaridad = 0.1139)
  5. key  (similaridad = 0.1078)

Palabra: 'team'
  1. teams  (similaridad = 0.2640)
  2. nhl  (similaridad = 0.2331)
  3. player  (similaridad = 0.2230)
  4. players  (similaridad = 0.2001)
  5. season  (similari

Interpretación palabra por palabra:

* university es el caso que más mejoró: pasó de 5 empates sin sentido a vecinos totalmente coherentes: professor, department, edu (el dominio de mail académico), boulder, urbana (ambas ciudades universitarias reales: Boulder es Colorado, Urbana es Illinois). Esto confirma la sospecha anterior: esas palabras "empatadas" eran justamente firmas de universidades que compartían un puñado de documentos, y al exigir min_df=10 esos términos únicos desaparecieron y quedaron los que genuinamente co-ocurren con university en muchos documentos.
* team se mantiene sólido y hasta mejora: ahora incluye season, sumando coherencia temática deportiva.
* computer mejoró bastante: graphics, drive, reboot son claramente términos de informática/hardware. wu e incremental son menos interpretables, pero ya no hay stop words ni ruido puro.
* person: people tiene sentido, pero el resto (presumed, incapable, smoked, implicitly) siguen sin relación clara. Esto sugiere que person es una palabra más "genérica" que se usa en muchísimos contextos distintos (argumentación, debates religiosos, políticos, etc.), por lo que no tiene un anclaje temático fuerte como sí lo tienen team o university.
* message: error, encrypted, key tienen sentido en contexto de informática/criptografía (mensajes cifrados, claves), aunque neglected y kindly no encajan. Similar a person, es una palabra que aparece en múltiples contextos (mensajes de foro en general, criptografía, errores de sistema), así que su "vecindario" es más heterogéneo.

Conclusión general del punto 4: el filtrado de vocabulario (stop words + min_df) es tan importante para similaridad de palabras como lo fue para el modelo de clasificación del punto 3. Además, se observa que palabras con significado más "acotado" y específico de un dominio (team, university) generan vecindarios semánticamente más limpios que palabras genéricas de uso transversal (person, message), que aparecen en muchos contextos distintos del corpus.